# Article extraction — scored against the gold article list

The frozen pipeline's output on the **held-out test volumes** (Penny Magazine 1836, Saturday Magazine 1840,
Chambers's Edinburgh Journal 1840 — three issues each, never used to write rules) scored against
`gold_article_list_test.csv`, which Owen annotated from the printed pages alone, without seeing the output
(guide: `documentation/article_extraction_pipeline_iterative_build/gold_article_list_annotation_guide.md`).

**How items are matched.** Each gold item is located in the output by its first and last words
(OCR-tolerant matching on the item's own pages). Every number below is computed by `score_gold.py`; nothing is
judged by hand.

| Outcome | Meaning |
|---|---|
| **exact** | the item opens a pipeline article of its own, and its last words are in that article |
| **merged** | its pipeline article also holds the start of the gold item before or after it (a missed boundary) |
| **split** | its first and last words land in different pipeline articles (a false boundary inside it) |
| **lost** | its first words are in no article's text (set aside as furniture, absorbed into other matter, or not found) |

- **Boundary recall** — gold items that open their own pipeline article. **Boundary precision** — pipeline
  articles that hold a gold item's start; the rest are *extra* pieces (fragments of split items, masthead lines…).
- **Fields** are scored for items that open their own article, across all the pipeline articles the item runs
  through if it was split.

Set `VERSION` below to score another version against the same gold list.

In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 200)

EVAL_DIR = next(p for p in [Path("."), Path("evaluation/evaluating_v2_article_extraction")]
                if (p / "score_gold.py").exists())
sys.path.insert(0, str(EVAL_DIR.resolve()))
from score_gold import load_gold, load_pipeline, score, summary_tables, write, VOLUME_ALIASES

VERSION = "v2"
OUT_DIR = EVAL_DIR  # results: <VERSION>_gold_comp_eval_*.csv / _summary.md

## Run the scoring and save the results

In [2]:
gold = load_gold()
res = score(VERSION, gold)
summary_path = write(res, VERSION, OUT_DIR)
items, arts, pieces, imprints = res["items"], res["pipeline_articles"], res["pieces"], res["imprints"]
tables = summary_tables(res)
own = items[items.opens_own_article]
print(f"Gold list: {len(gold)} rows — {gold.article_type.value_counts().to_dict()}")
print(f"Scored {len(items)} gold items against {len(arts)} pipeline articles ({VERSION})")
print(f"Wrote {OUT_DIR}/{VERSION}_gold_comp_eval_*.csv and {summary_path.name}")

Gold list: 96 rows — {'article': 85, 'imprint': 9, 'editorial_notice': 2}
Scored 87 gold items against 123 pipeline articles (v2)
Wrote ./v2_gold_comp_eval_*.csv and v2_gold_comp_eval_summary.md


## 1. Headline: items and boundaries

In [3]:
tables['items']

,penny_1836,saturday_1840,chambers_1840,all
gold items,15,31,41,87
pipeline articles,17,36,70,123
exact,14/15 (93%),22/31 (71%),27/41 (66%),63/87 (72%)
merged,0/15 (0%),2/31 (6%),2/41 (5%),4/87 (5%)
split,1/15 (7%),4/31 (13%),11/41 (27%),16/87 (18%)
lost,0/15 (0%),3/31 (10%),1/41 (2%),4/87 (5%)
boundary recall,15/15 (100%),27/31 (87%),39/41 (95%),81/87 (93%)
boundary precision,15/17 (88%),27/36 (75%),39/70 (56%),81/123 (66%)
extra pipeline articles,2,9,31,42


In [4]:
# Item outcome counts per volume
pd.crosstab(items.volume, items.status, margins=True, margins_name="total")

status,exact,lost,merged,split,total
volume,,,,,
chambers_1840,27,1,2,11,41
penny_1836,14,0,0,1,15
saturday_1840,22,3,2,4,31
total,63,4,4,16,87


## 2. Successes — items extracted exactly

In [5]:
exact = items[items.status == "exact"]
print(f"{len(exact)} of {len(items)} gold items extracted exactly")
exact[["volume", "gold_id", "gold_title", "pipeline_title", "start_article_id"]]

63 of 87 gold items extracted exactly


,volume,gold_id,gold_title,pipeline_title,start_article_id
0,penny_1836,1,The Giraffes at the Zoological Gardens,,penny_1836_0001
1,penny_1836,2,Working Men.,Working Men,penny_1836_0002
2,penny_1836,3,Threshing Machines.,Threshing Machines,penny_1836_0003
3,penny_1836,4,The Glyptothek.,NATIONAL GALLERIES FOR THE FINE ARTS AT MUNICH. THE GLYPTOTHEK,penny_1836_0004
5,penny_1836,6,War.,,penny_1836_0007
6,penny_1836,7,Importance of the Cultivation of the Vine in France.,IMPORTANCE OF THE CULTIVATION OF THE VINE IN FRANCE,penny_1836_0008
7,penny_1836,8,State of the Fine Arts in Geneva.,State of the Fine Arts in Geneva,penny_1836_0009
8,penny_1836,9,Division of Employments.,Division of Employments,penny_1836_0010
9,penny_1836,11,Rievaulx Abbey.,RIEVAULX ABBEY,penny_1836_0011
10,penny_1836,12,Manners of the Northern Coal Miners.,MANNERS OF THE NORTHERN COAL MINERS,penny_1836_0012


## 3. Errors — merged items (missed boundaries)

In [6]:
items[items.status.str.startswith("merged")][
    ["volume", "gold_id", "gold_title", "merged_with", "start_article_id", "pipeline_title"]]

,volume,gold_id,gold_title,merged_with,start_article_id,pipeline_title
27,saturday_1840,32,I. Nature of Geology,33,saturday_1840_0016,I. NATURE OF GEOLOGY
28,saturday_1840,33,,32,saturday_1840_0016,
84,chambers_1840,93,Natural Curiosity.,94,chambers_1839_0070,NATURAL CURIOSITY
85,chambers_1840,94,,93,chambers_1839_0070,


## 4. Errors — split items (false boundaries inside an item)

In [7]:
split = items[items.status.str.contains("split")]
split[["volume", "gold_id", "gold_title", "start_article_id", "end_article_id", "pipeline_articles"]]

,volume,gold_id,gold_title,start_article_id,end_article_id,pipeline_articles
4,penny_1836,5,Arts of Ancient Egypt - Glass and Porcelain.,penny_1836_0005,penny_1836_0006,penny_1836_0005;penny_1836_0006
19,saturday_1840,23,The Duchy of Saxe-Coburg-Gotha,saturday_1840_0005,saturday_1840_0007,saturday_1840_0005;saturday_1840_0006;saturday_1840_0007
31,saturday_1840,36,,saturday_1840_0020,saturday_1840_0021,saturday_1840_0020;saturday_1840_0021
40,saturday_1840,46,"Richmond Castle, Yorkshire.",saturday_1840_0030,saturday_1840_0031,saturday_1840_0030;saturday_1840_0031
43,saturday_1840,49,On Garden Herbs. I.,saturday_1840_0033,saturday_1840_0034,saturday_1840_0033;saturday_1840_0034
46,chambers_1840,53,Living in Paris,chambers_1839_0001,chambers_1839_0004,chambers_1839_0001;chambers_1839_0002;chambers_1839_0003;chambers_...
47,chambers_1840,54,Hibernation or Winter Sleep of Animals.,chambers_1839_0005,chambers_1839_0006,chambers_1839_0005;chambers_1839_0006
48,chambers_1840,55,Fugitive Writing of James Boswell,chambers_1839_0007,chambers_1839_0012,chambers_1839_0007;chambers_1839_0008;chambers_1839_0009;chambers_...
52,chambers_1840,59,Quack Advertisements.,chambers_1839_0020,chambers_1839_0021,chambers_1839_0020;chambers_1839_0021
58,chambers_1840,66,Opposite Good Qualities Not to Be Expected.,chambers_1839_0028,chambers_1839_0030,chambers_1839_0028;chambers_1839_0029;chambers_1839_0030


## 5. Errors — lost items (opening words not in any article's text)

In [8]:
items[items.status == "lost"][["volume", "gold_id", "gold_title", "start_found_in", "start_score"]]

,volume,gold_id,gold_title,start_found_in,start_score
18,saturday_1840,22,,box role: furniture,1.00
39,saturday_1840,45,,box role: furniture,0.86
42,saturday_1840,48,,box role: furniture,1.00
86,chambers_1840,95,,box role: furniture,0.86


## 6. Extra pipeline articles (hold no gold item's start)

Usually fragments of split items (`gold_ends` shows a gold item whose *last* words fall in it) or page
furniture taken as an article (masthead lines, prices, dates).

In [9]:
extra = arts[arts.status == "extra"]
print(extra.volume.value_counts().to_dict())
extra[["volume", "article_id", "pipeline_title", "word_count", "gold_ends", "text_start"]]

{'chambers_1839': 31, 'saturday_1840': 9, 'penny_1836': 2}


,volume,article_id,pipeline_title,word_count,gold_ends,text_start
5,penny_1836,penny_1836_0006,* Belzoni,710,5,"this substance is either obsidian or glass, and that it appears to"
15,penny_1836,penny_1836_0016,THE TOWER OF LONDON,4,,THE TOWER OF LONDON.
19,saturday_1840,saturday_1840_0003,THE CALENDAR,0,,
22,saturday_1840,saturday_1840_0006,GOTHA,1038,,"country, justice is administered by bailiffs, magistrates, and pat..."
23,saturday_1840,saturday_1840_0007,The house of Saxe,277,23,The house of Saxe-Coburg is indisputably the most fortunate of all...
28,saturday_1840,saturday_1840_0012,THE NATURAL HISTORY OF THE MONTHS. I. JANUARY,59,,"Then came old January, wrapped well In many weeds to keep the"
33,saturday_1840,saturday_1840_0017,NINEVEН,128,,"THE Lord will stretch out his hand against the north, and destroy"
37,saturday_1840,saturday_1840_0021,NaN,4,36,resignation to His will.
43,saturday_1840,saturday_1840_0027,NaN,22,,golf feddons Lendoval nest to dadifod sistibim tumpai os fro egaui...
47,saturday_1840,saturday_1840_0031,"RICHMOND CASTLE, YORKSHIRE",642,46,gate. There is a conducte of water at the Grey Freres els


## 7. Fields — summary

In [10]:
tables['fields']

,penny_1836,saturday_1840,chambers_1840,all
items scored (open their own article),15,27,39,81
title,12/15 (80%),21/27 (78%),33/39 (85%),66/81 (81%)
front matter pieces in place,1/3 (33%),0/4 (0%),2/8 (25%),3/15 (20%)
back matter pieces in place,9/12 (75%),10/15 (67%),22/37 (59%),41/64 (64%)
series (where either has one),0/3 (0%),0/6 (0%),6/16 (38%),6/25 (24%)
continued from,14/15 (93%),27/27 (100%),37/39 (95%),78/81 (96%)
to be continued,14/15 (93%),23/27 (85%),38/39 (97%),75/81 (93%)
editorial notice / article,15/15 (100%),27/27 (100%),39/39 (100%),81/81 (100%)
illustration count,11/15 (73%),24/27 (89%),39/39 (100%),74/81 (91%)
issue number,1/15 (7%),20/27 (74%),0/39 (0%),21/81 (26%)


### 7a. Titles that disagree

In [11]:
own[~own.title_result.isin(["correct", "both untitled"])][
    ["volume", "gold_id", "gold_title", "pipeline_title", "title_result"]]

,volume,gold_id,gold_title,pipeline_title,title_result
0,penny_1836,1,The Giraffes at the Zoological Gardens,,missing
5,penny_1836,6,War.,,missing
14,penny_1836,17,The Tower of London. - No. 1.,,missing
17,saturday_1840,21,The Calendar,,missing
24,saturday_1840,28,I. January,,missing
26,saturday_1840,31,The Timber-Floats of Germany and Italy,FLOATING TIMBER IN LOMBARDY,wrong
29,saturday_1840,34,Nineveh.,,missing
37,saturday_1840,43,"St. Omer’s, in France. I.","RUINS OF THE ABBEY OF ST. BERTIN, AT ST",wrong
43,saturday_1840,49,On Garden Herbs. I.,INTRODUCTION,wrong
51,chambers_1840,58,The Heir of the St Gerans.,STORY FROM THE CAUSES CELEBRES,wrong


### 7b. Front and back matter pieces

Each gold piece is found **in place** (the pipeline's front/back matter), **in text** (left in the article text),
elsewhere, or **missing** (set aside or not found). Footnotes — many in Chambers — are counted separately; some
give sources, others explain a word or concept.

In [12]:
tables['pieces']

result,in place,in text,missing,total
kind,,,,
back matter,16,12,0,28
back matter (footnote),25,4,7,36
front matter,3,7,5,15
total,44,23,12,79


In [13]:
# Pieces not in place
pieces[pieces.result != "in place"][["volume", "gold_id", "field", "footnote", "result", "piece"]]

,volume,gold_id,field,footnote,result,piece
0,penny_1836,2,back_matter,False,in text,"Dr F. Lieber, Plan of Education for Girard College of Orphans"
2,penny_1836,5,front_matter,False,in text,"A(bridged from ‘Egyptian Antiquities,’ vol. ii)"
3,penny_1836,5,back_matter,True,in text,*Belzoni
10,penny_1836,15,front_matter,False,in text,(From Sharon Turner’s ‘History of Henry VIII.’)
11,penny_1836,17,back_matter,True,in text,* Paul Hentzner’s ‘Journey into England in the year 1598
16,saturday_1840,21,front_matter,False,missing,180. 1. The Golden Number . . . 17 2. Epact . . . . . 26 3. Solar ...
17,saturday_1840,23,back_matter,False,in text,"From Germany; the Spirit of her History, Literature, Social Condit..."
21,saturday_1840,28,front_matter,False,missing,"Then came old January, wrapped all In many weeds to keep the cold ..."
24,saturday_1840,32,back_matter,True,in text,"* Lyell’s Anniversary Address, 1837"
26,saturday_1840,34,front_matter,False,missing,"The Lord will stretch out his hand against the north, and destroy ..."


In [14]:
# Pipeline front/back matter with no gold counterpart (extra pieces), per item
x = own[(own.front_matter_extra > 0) | (own.back_matter_extra > 0)]
x[["volume", "gold_id", "gold_title", "front_matter_extra", "front_matter_pipeline",
   "back_matter_extra", "back_matter_pipeline"]]

,volume,gold_id,gold_title,front_matter_extra,front_matter_pipeline,back_matter_extra,back_matter_pipeline
19,saturday_1840,23,The Duchy of Saxe-Coburg-Gotha,0.0,,1.0,* The husband-elect of our gracious sovereign Queen Victoria. Ev. ...
48,chambers_1840,55,Fugitive Writing of James Boswell,0.0,,3.0,* Irritability is a term used by anatomists to indicate that prope...
52,chambers_1840,59,Quack Advertisements.,0.0,,1.0,Glasgow Constitutional
60,chambers_1840,68,Alchemy.,0.0,,1.0,"* Retrospective Review, vol. xiv, article Alchemy. | * Her eulogy ..."
63,chambers_1840,71,M. Turgot.,0.0,,1.0,* Sir Walter's brother Thomas was married to a sister of Mr M'Cull...
78,chambers_1840,87,The Barber’s Apprentice.,0.0,,1.0,* The robbery was committed about two o'clock on the morning of th...
80,chambers_1840,89,Speculation on Words.,0.0,,1.0,* Captain Marryat alleges that the Americans who visit England app...
81,chambers_1840,90,Scandinavian Mythology.,0.0,,1.0,* We use the translation of the story in the Foreign Quarterly Rev...


### 7c. Series and continuations that disagree

In [15]:
g = gold.set_index(["volume", "gold_id"])
s = own[~own.series_result.isin(["correct", "none (both)"])].copy()
s["gold_series"] = [g.loc[(v, i), "series"] for v, i in zip(s.volume, s.gold_id)]
s[["volume", "gold_id", "gold_title", "gold_series", "pipeline_series", "series_result"]]

,volume,gold_id,gold_title,gold_series,pipeline_series,series_result
3,penny_1836,4,The Glyptothek.,National Galleries for the Fine Arts at Munich.,,missing
11,penny_1836,13,The Pinacothek.,National Galleries for the Fine Arts at Munich.,,missing
12,penny_1836,14,Knights of Malta. - No. III.,,Knights of Malta No,extra
24,saturday_1840,28,I. January,The Natural History of the Months.,,missing
26,saturday_1840,31,The Timber-Floats of Germany and Italy,,The Timber Floats of Germany and Italy,extra
27,saturday_1840,32,I. Nature of Geology,,A Discourse on Geology,extra
37,saturday_1840,43,"St. Omer’s, in France. I.",,One Panny,extra
38,saturday_1840,44,On Military Drilling. I.,,On Military Drilling,extra
43,saturday_1840,49,On Garden Herbs. I.,,On Garden Herbs I,extra
46,chambers_1840,53,Living in Paris,,Number,extra


In [16]:
own[~own.continued_from_result.str.startswith("correct") | ~own.to_be_continued_result.str.startswith("correct")][
    ["volume", "gold_id", "gold_title", "continued_from_result", "to_be_continued_result"]]

,volume,gold_id,gold_title,continued_from_result,to_be_continued_result
12,penny_1836,14,Knights of Malta. - No. III.,missing,correct (no)
14,penny_1836,17,The Tower of London. - No. 1.,correct (no),missing
15,saturday_1840,19,Sir Joshua Reynolds and His Works. I.,correct (no),missing
37,saturday_1840,43,"St. Omer’s, in France. I.",correct (no),missing
38,saturday_1840,44,On Military Drilling. I.,correct (no),missing
43,saturday_1840,49,On Garden Herbs. I.,correct (no),missing
63,chambers_1840,71,M. Turgot.,correct (no),missing
73,chambers_1840,82,M. Turgot.,missing,correct (no)
80,chambers_1840,89,Speculation on Words.,missing,correct (no)


### 7d. Illustrations and article type that disagree

In [17]:
own[own.figures_result != "correct"][["volume", "gold_id", "gold_title", "figures_gold", "figures_pipeline"]]

,volume,gold_id,gold_title,figures_gold,figures_pipeline
0,penny_1836,1,The Giraffes at the Zoological Gardens,1.0,0.0
11,penny_1836,13,The Pinacothek.,2.0,1.0
12,penny_1836,14,Knights of Malta. - No. III.,0.0,1.0
14,penny_1836,17,The Tower of London. - No. 1.,1.0,2.0
15,saturday_1840,19,Sir Joshua Reynolds and His Works. I.,1.0,0.0
29,saturday_1840,34,Nineveh.,1.0,0.0
37,saturday_1840,43,"St. Omer’s, in France. I.",1.0,0.0


In [18]:
own[~own.type_result.str.startswith("correct")][["volume", "gold_id", "gold_title", "article_type", "type_result"]]

,volume,gold_id,gold_title,article_type,type_result


## 8. Issue metadata — number and date

In [19]:
issue = own.groupby(["volume", "issue_no"]).agg(
    pipeline_issue_no=("pipeline_issue_no", lambda x: ", ".join(sorted(set(map(str, x))))),
    pipeline_issue_date=("pipeline_issue_date", lambda x: ", ".join(sorted(set(map(str, x))))),
    number_correct=("issue_no_result", lambda x: (x == "correct").mean().round(2)),
    date_correct=("issue_date_result", lambda x: (x == "correct").mean().round(2)))
issue

pipeline_issue_no pipeline_issue_date  number_correct  date_correct
volume        issue_no                                                                    
chambers_1840 414                                                        0.0           0.0
              415                           January 25, 1746             0.0           0.0
              416                                                        0.0           0.0
penny_1836    270                              June 18, 1836             0.0           1.0
              271                     10       June 25, 1836             0.0           1.0
              272                    272        May 31, 1836             1.0           0.0
saturday_1840 482                    482     January 4, 1840             1.0           1.0
              483                    483    January 11, 1840             1.0           1.0
              484                           January 18, 1840             0.0           1.0

## 9. Imprints (publishing history)

In [20]:
if "imprints" in tables:
    display(tables["imprints"])
imprints[["volume", "gold_id", "issue_no", "result", "missing_part_found_in", "pipeline_imprint"]]

result,complete,total
volume,,
chambers_1840,3,3
penny_1836,3,3
saturday_1840,3,3
total,9,9


,volume,gold_id,issue_no,result,missing_part_found_in,pipeline_imprint
0,penny_1836,10,270,complete,,The Office of the Society for the Diffusion of Useful Knowledge is...
1,penny_1836,16,271,complete,,The Office of the Society for the Diffusion of Useful Knowledge is...
2,penny_1836,18,272,complete,,The Office of the Society for the Diffusion of Useful Knowledge is...
3,saturday_1840,30,482,complete,,"LONDON: | JOHN WILLIAM PARKER, WEST STRAND. | PUBLISHED IN WEEKLY ..."
4,saturday_1840,42,483,complete,,"LONDON: | JOHN WILLIAM PARKER, WEST STRAND. | PUBLISHED IN WEEKLY ..."
5,saturday_1840,52,484,complete,,"LONDON: JOHN WILLIAM PARKER, WEST STRAND. PUBLISHED IN WEEKLY NUMB..."
6,chambers_1840,65,414,complete,,"LONDON: Published, with permission of the proprietors, by W.S. ORR..."
7,chambers_1840,80,415,complete,,"LONDON: Published, with permission of the proprietors, by W.S. ORR..."
8,chambers_1840,98,416,complete,,"LONDON: Published, with permission of the Proprietors, by W. S. OR..."


## 10. Look up one gold item

In [21]:
def show(volume, gold_id):
    """A gold row next to what the pipeline produced for it."""
    row = gold[(gold.volume == volume) & (gold.gold_id == gold_id)].iloc[0]
    it = items[(items.volume == volume) & (items.gold_id == gold_id)]
    print("GOLD:", {k: v for k, v in row.items() if v and k not in ("pipeline_volume",)})
    if len(it):
        it = it.iloc[0]
        print("\nRESULT:", it.status, "| title:", it.get("title_result", ""), "| merged with:", it.merged_with or "-")
        pipe = load_pipeline(VERSION, VOLUME_ALIASES.get(volume, volume))["articles"]
        ids = str(it.get("pipeline_articles") or it.start_article_id or "").split(";")
        for aid in [a for a in ids if a in pipe.index]:
            a = pipe.loc[aid]
            print(f"\n{aid}: {a.title!r}  pages {a.pages}")
            print("  front:", a.front_matter if isinstance(a.front_matter, str) else "-")
            print("  text :", " ".join(str(a.text).split()[:30]), "…")
            print("  back :", a.back_matter if isinstance(a.back_matter, str) else "-")

show("chambers_1840", 58)

GOLD: {'gold_id': 58, 'volume': 'chambers_1840', 'issue_no': '414', 'issue_date': '1.4.1840', 'start_pdf_page': 12, 'start_printed_page': '398', 'start_column': 'right', 'end_pdf_page': '14', 'article_type': 'article', 'title': 'The Heir of the St Gerans.', 'first_words': 'The Marshal de St', 'last_words': 'of conduct they pursued.', 'front_matter': 'Story from the Causes Celebres.', 'back_matter': '* The original narrative admits the difficulty which arises here on account of the milk, which it would be necessary to conceal. There are two ways of explaining this point. It is possible that the case of the countess, happily for the plotters, was one in which the lacteal supply was naturally deficient; and it is also possible, as is suggested in the text, that the nurse may have been skillful enough to check it by the use of drugs, locally applied as well as internally. Had the countess called in a medical man, the trick must have been discovered. To account for her not doing so, we must